### 1-Setup

In [14]:
import os
import numpy as np
import pandas as pd

for d in ("data/raw", "data/own", "data/processed", "figures"):
    os.makedirs(d, exist_ok=True)

### 2-IMDB 50K

In [ ]:


path = "data/raw/imdb_50k.csv"

if not os.path.exists(path):
    url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
    imdb = pd.read_csv(url)
    imdb["label"] = (imdb["sentiment"] == "positive").astype(int)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    imdb.to_csv(path, index=False)

imdb = pd.read_csv(path)

if "label" not in imdb.columns:
    imdb["label"] = (imdb["sentiment"] == "positive").astype(int)
    imdb.to_csv(path, index=False)

print(imdb.shape, imdb["label"].value_counts().to_dict())

### Pang & Lee

In [ ]:
import nltk
from nltk.corpus import movie_reviews

path = "data/raw/pang_lee_2000.csv"
if not os.path.exists(path):
    nltk.download("movie_reviews", quiet=True)
    rows = [(movie_reviews.raw(f), int(c == "pos"))
            for c in movie_reviews.categories()
            for f in movie_reviews.fileids(c)]
    pd.DataFrame(rows, columns=["review", "label"]).to_csv(path, index=False)
pl = pd.read_csv(path)
print(pl.shape, pl["label"].value_counts().to_dict(), round(pl["review"].str.split().str.len().mean(), 1))

In [ ]:
old, new = "data/own/tmdb_raw.csv", "data/own/tmdb_raw_pool1.csv"
if os.path.exists(old) and not os.path.exists(new):
    os.rename(old, new)
print(os.path.exists(new), os.path.exists(old))

### 3-TMDB

In [ ]:
import time, requests
from getpass import getpass

BASE = "https://api.themoviedb.org/3"
KEY = os.environ.get("TMDB_KEY")
GENRES = {"Action": 28, "Comedy": 35, "Drama": 18, "Horror": 27,
          "Sci-Fi": 878, "Animation": 16, "Romance": 10749, "Thriller": 53}

def get(path, **params):
    for attempt in range(4):
        try:
            r = requests.get(f"{BASE}{path}", params={"api_key": KEY, **params}, timeout=20)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        if r.status_code == 401:
            raise RuntimeError("Invalid TMDB API key (401)")
        if r.status_code == 429:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
        time.sleep(0.15)
        return r.json()
    return {"results": []}

def collect(extra, pages=4, start_page=1):
    rows = []
    for genre, gid in GENRES.items():
        for page in range(start_page, start_page + pages):
            res = get("/discover/movie", with_genres=gid, page=page, language="en-US",
                      **{"primary_release_date.gte": "2016-01-01"}, **extra)["results"]
            for m in res:
                data = get(f"/movie/{m['id']}/reviews", language="en-US")
                for rv in data.get("results", []):
                    rating = (rv.get("author_details") or {}).get("rating")
                    if rating is None:
                        continue
                    rows.append({"review": rv["content"], "rating": rating,
                                 "film": m["title"], "year": (m.get("release_date") or "")[:4],
                                 "genre": genre, "source": "TMDB", "review_id": rv["id"]})
        print(genre, len(rows))
    return pd.DataFrame(rows)

### collect with genres and low-rated films

In [ ]:
raw_path = "data/own/tmdb_raw.csv"

if os.path.exists(raw_path):
    tmdb = pd.read_csv(raw_path)
else:
    high = collect({"sort_by": "vote_count.desc"})
    low = collect({"sort_by": "vote_average.asc", "vote_count.gte": 300})
    tmdb = pd.concat([high, low]).drop_duplicates(subset="review_id")
    tmdb.to_csv(raw_path, index=False)

print(tmdb.shape)
print("rating >= 7:", int((tmdb["rating"] >= 7).sum()), "| rating <= 4:", int((tmdb["rating"] <= 4).sum()))

### merge the two pools 

In [ ]:
pool1 = pd.read_csv("data/own/tmdb_raw_pool1.csv")
pool2 = pd.read_csv("data/own/tmdb_raw.csv")
tmdb = pd.concat([pool2, pool1]).drop_duplicates(subset="review_id").reset_index(drop=True)
tmdb.to_csv("data/own/tmdb_raw.csv", index=False)
print(tmdb.shape, tmdb["genre"].value_counts().to_dict())
print("rating >= 7:", int((tmdb["rating"] >= 7).sum()), "| rating <= 4:", int((tmdb["rating"] <= 4).sum()))

### top up negatives

In [ ]:
more = collect({"sort_by": "vote_average.asc", "vote_count.gte": 100}, pages=4, start_page=5)
tmdb = pd.concat([tmdb, more]).drop_duplicates(subset="review_id")
tmdb.to_csv("data/own/tmdb_raw.csv", index=False)
print(tmdb.shape, "rating <= 4:", int((tmdb["rating"] <= 4).sum()))

### normalize , flag hard cases, filter

In [11]:
raw = pd.read_csv(raw_path)
df = raw.dropna(subset=["review", "rating", "film"]).copy()
df["review"] = df["review"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["scale"] = 10
df["rating10"] = df["rating"] * 10 / df["scale"]
df["n_words"] = df["review"].str.split().str.len()
df["ascii_share"] = df["review"].apply(lambda s: sum(ord(c) < 128 for c in s) / max(len(s), 1))

nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
df["vader"] = df["review"].apply(lambda s: sia.polarity_scores(s)["compound"])
df["hard_case"] = (((df["rating10"] >= 7) & (df["vader"] <= 0))
                   | ((df["rating10"] <= 4) & (df["vader"] > 0))).astype(int)

log = {}
count = len(df)

def step(name, mask):
    global df, count
    df = df[mask]
    log[name] = count - len(df)
    count = len(df)

step("duplicate text", ~df.duplicated(subset="review"))
step("not English-looking", df["ascii_share"] > 0.97)
step("outside 20-300 words", df["n_words"].between(20, 300))
step("film before 2016", df["year"] >= 2016)
step("neutral rating (between 4 and 7 out of 10)", (df["rating10"] >= 7) | (df["rating10"] <= 4))

df["label"] = (df["rating10"] >= 7).astype(int)
print(log)
print(len(df), df["label"].value_counts().to_dict())

{'duplicate text': 6, 'not English-looking': 9, 'outside 20-300 words': 1061, 'film before 2016': 0, 'neutral rating (between 4 and 7 out of 10)': 800}
2125 {1: 1491, 0: 634}


### inspect

In [12]:
print(df.groupby(["genre", "label"]).size().unstack(fill_value=0))
print(df["film"].nunique(), int(df["hard_case"].sum()))
print(df.groupby("film").size().sort_values(ascending=False).head(8))

label        0    1
genre              
Action     150  335
Animation   42  102
Comedy      52  214
Drama       75  176
Horror     121  216
Romance     35   82
Sci-Fi      59   63
Thriller    26   61
Various     74  242
656 484
film
Project Hail Mary                    13
Barbie                               13
Everything Everywhere All at Once    12
Captain Marvel                       12
Snow White                           12
Dune: Part Two                       12
John Wick: Chapter 4                 11
The Odyssey                          11
dtype: int64


### select the final 300 and crun the checks

In [15]:
def pick(g, n=150, per_film=6):
    g = g.sample(frac=1, random_state=42).groupby("film").head(per_film)
    known = g[g["genre"] != "Various"]
    quota = int(np.ceil(n / max(known["genre"].nunique(), 1)))
    first = known.groupby("genre").head(quota)
    rest = g.drop(first.index)
    return pd.concat([first, rest]).head(n)

own = pd.concat([pick(df[df.label == 1]), pick(df[df.label == 0])])
own = own.sample(frac=1, random_state=42).reset_index(drop=True)

final = {
    "300 reviews": len(own) == 300,
    "150 / 150": bool(own["label"].value_counts().eq(150).all()),
    "5+ genres": own.loc[own["genre"] != "Various", "genre"].nunique() >= 5,
    "40+ films": own["film"].nunique() >= 40,
    "max 6 per film": own.groupby("film").size().max() <= 6,
    "30+ hard cases": own["hard_case"].sum() >= 30,
}
print(pd.Series(final))
print("genre unknown:", int((own["genre"] == "Various").sum()))
print(own.groupby(["genre", "label"]).size().unstack(fill_value=0))

300 reviews       True
150 / 150         True
5+ genres         True
40+ films         True
max 6 per film    True
30+ hard cases    True
dtype: bool
genre unknown: 0
label       0   1
genre            
Action     19  19
Animation  19  19
Comedy     19  19
Drama      19  19
Horror     19  19
Romance    19  19
Sci-Fi     19  19
Thriller   17  17


### save and send to a friend 50 unlabeled reviews

In [16]:
cols = ["review", "label", "rating", "scale", "rating10", "source", "film", "genre", "year", "hard_case"]
own[cols].to_csv("data/own/own_reviews.csv", index=False)

sample = own.groupby("label").sample(25, random_state=7)
sample[["review"]].assign(label="").to_csv("data/own/to_annotate.csv")

## Summary

In [17]:
print(len(own), own["label"].value_counts().to_dict())
print(own["source"].value_counts().to_dict())
print(own["genre"].value_counts().to_dict())
print(own["film"].nunique(), int(own["year"].min()), int(own["year"].max()))
print("hard cases:", int(own["hard_case"].sum()), own.groupby("label")["hard_case"].sum().to_dict())
print(round(own["review"].str.split().str.len().mean(), 1))

300 {0: 150, 1: 150}
{'TMDB': 300}
{'Sci-Fi': 38, 'Drama': 38, 'Horror': 38, 'Comedy': 38, 'Romance': 38, 'Animation': 38, 'Action': 38, 'Thriller': 34}
215 2016 2026
hard cases: 95 {0: 78, 1: 17}
126.8


### Compute the cappa

In [18]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

own = pd.read_csv("data/own/own_reviews.csv")
ann = pd.read_csv("data/own/annotated.csv", index_col=0).dropna(subset=["label"])
y_true = own.loc[ann.index, "label"]
y_ann = ann["label"].astype(int)
print(len(ann), round(cohen_kappa_score(y_true, y_ann), 3))
print(confusion_matrix(y_true, y_ann))

for _, r in own.loc[ann.index][y_true != y_ann].iterrows():
    print(r["label"], r["rating10"], "|", r["review"][:250], "\n")

50 0.96
[[24  1]
 [ 0 25]]
0 4.0 | This was actually quite funny and didn't take any prisoners. As far as cheap streaming comedies go, this one delivered lots of entertainment. 

